In [15]:
import numpy as np
import rasterio
import joblib
import gc
import os
import pandas as pd
import glob
import geopandas as gpd

from maxent_model import load_model_config
from osgeo import gdal
from rasterio.windows import Window
from shapely.geometry import mapping
from rasterio.mask import mask
from shapely.geometry import shape
from rasterio import features

In [16]:
sp = 'protonotaria_citrea'
tp = 'spring'
basedir = '/mnt/f/readyparams'

In [17]:
sp = sp.lower()
spdir = os.path.join(basedir,f'ppp_paramsoutput/{sp}')

raster_path = os.path.join(basedir, 'rasters', 'inference')
origrasterpath = raster_path

print(spdir)

sp = sp.lower()
spdir = os.path.join(basedir, f'ppp_paramsoutput/{sp}')
model_path   = os.path.join(spdir, f"elapid_maxent_model_tuned_{sp}.pkl")     # tuned elapid model
metrics_path = os.path.join(spdir, f"accuracy_tuned_{sp}.csv")                # tuned metrics (includes BestThreshold)
band_source_dir = os.path.join(basedir, 'rasters', 'inference')                # where a .tif has band descriptions
rangefile =  os.path.join(spdir, f"convex_hull_{sp}.json")

model_config = load_model_config(spdir, sp)
training_columns = model_config['predictor_columns']
excel_group = model_config.get('excel_group', 'birds')
categorical_features = model_config.get('categorical_features', [])
print(f"Inference group: {excel_group}, predictors: {len(training_columns)}")

output_prob_path = os.path.join(spdir, f"predictions_prob_{sp}_{tp}.tif")
output_bin_path  = os.path.join(spdir, f"predictions_binary_{sp}_{tp}.tif")


/mnt/f/readyparams/ppp_paramsoutput/protonotaria_citrea


In [18]:
def build_mosaic_vrt_in_memory(raster_dir, rangefile_path, pattern=f"all_months*.tif", tp=tp):
    # Find the two input rasters
    tpraster = f'{tp}.tif'
    rasters = sorted(glob.glob(os.path.join(raster_dir, pattern)))
    print(rasters)
    # Build an in-memory VRT (mosaic)
    # Use separate=False so overlapping areas are blended; you can also set options like 'resolution', 'resampling'
    vrt_path = f"/vsimem/{sp}_{tp}_mosaic.vrt"
    vrt = gdal.BuildVRT(
        vrt_path,
        rasters,
        separate=False
    )
    if vrt is None:
        raise RuntimeError("BuildVRT failed.")
    vrt.FlushCache()
    
    # 3) Path to all.tif (the band stack to append)
    all_tif_path = os.path.join(raster_dir, tpraster)
    if not os.path.exists(all_tif_path):
        raise FileNotFoundError(f"Could not find {all_tif_path}")
    stacked_vrt_path = f"/vsimem/{sp}_{tp}_mosaic_plus_all.vrt"
    stacked_vrt = gdal.BuildVRT(
        stacked_vrt_path,
        [vrt_path, all_tif_path],
        separate=True
    )
    stacked_vrt.FlushCache()

     # 3) Cast all bands to Float32 by translating to a new VRT
    cast_vrt_path = f"/vsimem/{sp}_{tp}_mosaic_plus_all_float32.vrt"
    translate_opts = gdal.TranslateOptions(format="VRT", outputType=gdal.GDT_Float32)
    cast_vrt = gdal.Translate(cast_vrt_path, stacked_vrt, options=translate_opts)
    if cast_vrt is None:
        raise RuntimeError("Translate to Float32 VRT failed.")
    cast_vrt.FlushCache()
    
    warp_opts = gdal.WarpOptions(
    format="VRT",
    cutlineDSName=rangefile_path,
    cropToCutline=True,
    dstNodata=0
    )
    final_vrt_path = f"/vsimem/{sp}_{tp}_final.vrt"
    # We warp the 'cast_vrt' so the output is already Float32
    final_vrt = gdal.Warp(final_vrt_path, cast_vrt, options=warp_opts)
    final_vrt.FlushCache()
    return final_vrt_path

In [19]:
def predict_raster_with_elapid(
    model_path: str,
    metrics_path: str,
    raster_path: str,
    rangefile_path: str,
    output_prob_path: str,
    output_bin_path: str,
    training_columns: list,          # <<< pass the raw X column names used at training
    band_source_dir: str,            # directory containing a representative .tif with band descriptions
    batch_size: int = 1000,
    nodata_value: float = -9999.0,
    log_every: int = 50,
    sample_per_tile: int = 1000,
    tp = tp
):
    """
    Apply a tuned elapid MaxEnt model over a raster stack to produce probability & binary maps.
    - Assumes the model was saved via joblib as elapid_maxent_model_tuned_{sp}.pkl
    - Reads the tuned 'BestThreshold' from accuracy_tuned_{sp}.csv
    - Expects 'raster_path' to have the same bands (by name) as 'training_columns'.
    """

    # ------------------------
    # Load tuned model & metrics
    # ------------------------
    model = joblib.load(model_path)
    metrics = pd.read_csv(metrics_path, header=None, index_col=0)

    # We saved 'BestThreshold' in training; use that for binarization
    if 'BestThreshold' in metrics.index:
        best_threshold = float(metrics.loc['BestThreshold'][1])
    elif 'Threshold' in metrics.index:  # fallback if you kept old column name
        best_threshold = float(metrics.loc['Threshold'][1])
    else:
        raise ValueError("BestThreshold not found in metrics CSV; please include it when saving tuned metrics.")

    print(f"✅ Loaded tuned elapid model and metrics: threshold={best_threshold:.4f}")

    # ------------------------
    # Map training columns to raster band descriptions
    # ------------------------
    # We use the first raster in a folder to read band descriptions (GDAL GetDescription),
    # then build a mapping from training column -> band index (1-based for GDAL/rasterio).
    # This lets us reorder raster bands to match training column order.
    # If descriptions are empty, consider passing a manual alias dict or ensure descriptions are set.
    first_tif = sorted(glob.glob(os.path.join(band_source_dir, "*all_months*.tif")))
    print('Spectral tifs:', first_tif)
    if not first_tif:
        raise FileNotFoundError(f"No .tif files found in {band_source_dir} to read band descriptions.")
    ds = gdal.Open(first_tif[0])
    raster_band_names = [ds.GetRasterBand(i).GetDescription() for i in range(1, ds.RasterCount + 1)]
    print("✅ Raster band names (GDAL GetDescription):", raster_band_names)

    env_tif = sorted(glob.glob(os.path.join(band_source_dir, f"*{tp}.tif")))
    print('Env tif:', env_tif)
    if not env_tif:
        raise FileNotFoundError(f"No .tif files found in {band_source_dir} to read band descriptions.")
    ds = gdal.Open(env_tif[0])
    env_band_names = [ds.GetRasterBand(i).GetDescription() for i in range(1, ds.RasterCount + 1)]
    print("✅ Raster env band names (GDAL GetDescription):", env_band_names)
    raster_band_names = raster_band_names + env_band_names
    # case-insensitive matching helper
    def norm(s): return (s or "").strip().lower()

    # Build mapping: feature -> band index
    band_mapping = {}
    print(training_columns)
    for feature in training_columns:
        matches = [i for i, nm in enumerate(raster_band_names) if norm(nm) == norm(feature)]
        if not matches:
            # Relaxed match: underscore/space-insensitive
            f_alt = norm(feature).replace("_","").replace(" ","")
            for i, nm in enumerate(raster_band_names):
                nm_alt = norm(nm).replace("_","").replace(" ","")
                if nm_alt == f_alt:
                    matches = [i]
                    break
        if not matches:
            raise ValueError(f"❌ Feature '{feature}' not found among raster band descriptions. "
                             f"Set band descriptions to match training column names or pass a manual mapping.")
        band_mapping[feature] = matches[0] + 1  # 1-based for rasterio's indexes
    print("✅ Band mapping created:", band_mapping)

    band_indices = [band_mapping[f] for f in training_columns]
    

    # ------------------------
    # Open raster and validate
    # ------------------------
    with rasterio.open(raster_path) as src:
        n_bands = src.count
        if n_bands != len(training_columns):
            #raise ValueError(f"Raster has {n_bands} bands, but model expects {len(training_columns)} features.")
            print(f"Raster has {n_bands} bands, but model expects {len(training_columns)} features.")

        print(f"✅ Using band indices in training order: {band_indices}")
        gdf_hull = gpd.read_file(rangefile_path).to_crs(src.crs)
        hull_geom = [mapping(gdf_hull.geometry.unary_union)]
        # Output profile
        profile = src.profile.copy()
        profile.update(driver='GTiff', dtype='float32', count=1, compress='lzw',
                       tiled=True, blockxsize=128, blockysize=128, nodata=nodata_value)

        tile_idx = 0
        total_valid = 0
        total_nan = 0
        total_above_thr = 0
        sampled_probs = []

        with rasterio.open(output_prob_path, 'w', **profile) as dst_prob, \
             rasterio.open(output_bin_path, 'w', **profile) as dst_bin:

            for _, window in src.block_windows(1):
                tile_idx += 1
                # Read & reorder bands for this tile
                data = src.read(indexes=band_indices, window=window, out_dtype='float32')
                h, w = data.shape[1], data.shape[2]
                window_transform = src.window_transform(window)
                mask = features.geometry_mask(
                    hull_geom, 
                    out_shape=(h, w), 
                    transform=window_transform, 
                    invert=True # True means pixels INSIDE the hull are True
                )
                # Flatten to N x p
                X = np.moveaxis(data, 0, -1).reshape(-1, len(training_columns))
                
                preds_prob = np.full(X.shape[0], np.nan, dtype=np.float32)
                preds_bin = np.full(X.shape[0], np.nan, dtype=np.float32)

                # Valid pixels = no NaNs across features
                spatial_valid = mask.flatten() 
                # Update 'valid' to only include pixels INSIDE the hull AND NOT NaN
                valid = (~np.isnan(X).any(axis=1)) & spatial_valid
                valid_idx = np.where(valid)[0]
                total_nan += (~valid).sum()
                total_valid += valid.sum()

                if valid_idx.size > 0:
                    # Batch prediction over valid pixels
                    for start in range(0, valid_idx.size, batch_size):
                        end = start + batch_size
                        batch_sel = valid_idx[start:end]
                        # pandas DataFrame preserves column names expected by elapid
                        X_batch = pd.DataFrame(X[batch_sel, :], columns=training_columns)
                        
                        # 🔑 elapid MaxEnt returns cloglog-calibrated probabilities directly (PPP-consistent)
                        prob = model.predict(X_batch).astype(np.float32)  # shape (n_batch,)

                        binary = (prob >= best_threshold).astype(np.float32)

                        preds_prob[batch_sel] = prob
                        preds_bin[batch_sel] = binary

                    if tile_idx % log_every == 0:
                        q = np.percentile(preds_prob[valid], [0, 25, 50, 75, 95, 99])
                        print(f"🧪 Tile {tile_idx}: prob pct [min,25,50,75,95,99] = {q}")

                    total_above_thr += np.nansum(preds_bin == 1)

                    # Sample probs for global summary
                    k = min(sample_per_tile, preds_prob[valid].size)
                    if k > 0:
                        sel = np.random.choice(preds_prob[valid].size, size=k, replace=False)
                        sampled_probs.extend(preds_prob[valid][sel].tolist())

                # Write blocks
                prob_block = np.where(np.isnan(preds_prob.reshape((h, w))), nodata_value, preds_prob.reshape((h, w)))
                bin_block = np.where(np.isnan(preds_bin.reshape((h, w))), nodata_value, preds_bin.reshape((h, w)))

                dst_prob.write(prob_block.astype(np.float32), indexes=1, window=window)
                dst_bin.write(bin_block.astype(np.float32), indexes=1, window=window)

                del data, X, preds_prob, preds_bin, prob_block, bin_block
                gc.collect()

    if sampled_probs:
        gq = np.percentile(np.array(sampled_probs, dtype=np.float32), [0, 25, 50, 75, 95, 99])
        print(f"📊 Global prob pct [min,25,50,75,95,99] = {gq}")

    print(f"✅ Probability raster saved to {output_prob_path}")
    print(f"✅ Binary raster saved to {output_bin_path}")


In [20]:
mosaic_vrt = build_mosaic_vrt_in_memory(raster_path, rangefile)
print(mosaic_vrt)
predict_raster_with_elapid(
    model_path=model_path,
    metrics_path=metrics_path,
    raster_path=mosaic_vrt,
    rangefile_path=rangefile,
    output_prob_path=output_prob_path,
    output_bin_path=output_bin_path,
    training_columns=training_columns,
    band_source_dir=band_source_dir,
    batch_size=2000,
    nodata_value=-9999.0,
    log_every=50,
    sample_per_tile=1000,
    tp=tp
)

['/mnt/f/readyparams/rasters/inference/all_months_2025-0000000000-0000000001.tif', '/mnt/f/readyparams/rasters/inference/all_months_2025-0000000000-0000005632.tif', '/mnt/f/readyparams/rasters/inference/all_months_2025-0000005632-0000000000.tif', '/mnt/f/readyparams/rasters/inference/all_months_2025-0000005632-0000005632.tif']
/vsimem/protonotaria_citrea_spring_final.vrt
✅ Loaded tuned elapid model and metrics: threshold=0.2060
Spectral tifs: ['/mnt/f/readyparams/rasters/inference/all_months_2025-0000000000-0000000001.tif', '/mnt/f/readyparams/rasters/inference/all_months_2025-0000000000-0000005632.tif', '/mnt/f/readyparams/rasters/inference/all_months_2025-0000005632-0000000000.tif', '/mnt/f/readyparams/rasters/inference/all_months_2025-0000005632-0000005632.tif']
✅ Raster band names (GDAL GetDescription): ['dw_class_0_pct_500m', 'dw_class_1_pct_500m', 'dw_class_2_pct_500m', 'dw_class_3_pct_500m', 'dw_class_4_pct_500m', 'dw_class_5_pct_500m', 'dw_class_6_pct_500m', 'dw_class_7_pct_500

/tmp/ipykernel_7321/4253795661.py:98: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hull_geom = [mapping(gdf_hull.geometry.unary_union)]


📊 Global prob pct [min,25,50,75,95,99] = [1.96138058e-06 4.53945138e-02 2.14270353e-01 5.25456637e-01
 9.95337266e-01 1.00000000e+00]
✅ Probability raster saved to /mnt/f/readyparams/ppp_paramsoutput/protonotaria_citrea/predictions_prob_protonotaria_citrea_spring.tif
✅ Binary raster saved to /mnt/f/readyparams/ppp_paramsoutput/protonotaria_citrea/predictions_binary_protonotaria_citrea_spring.tif
